In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
# IRN = Indice de regularidade normalizado

FILE_PATH = "/content/viagem_informada.csv" #



df = pd.read_csv("/content/viagem_informada.csv")

df["partida_dt"] = pd.to_datetime(df["partida"], format="%H:%M", errors="coerce")
df = df.dropna(subset=["partida_dt"])

df["hora"] = df["partida_dt"].dt.hour

df["data_dt"] = pd.to_datetime(df["data"], format="%d/%m/%Y", errors="coerce")
df = df.dropna(subset=["data_dt"])

df["minuto_dia"] = df["partida_dt"].dt.hour * 60 + df["partida_dt"].dt.minute


# Faixas horárias

FAIXAS = {
    "Madrugada (00h–05h)": (0, 5),
    "Manhã Pico (06h–09h)": (6, 9),
    "Manhã Fora-Pico (10h–11h)": (10, 11),
    "Almoço (12h–13h)": (12, 13),
    "Tarde Fora-Pico (14h–16h)": (14, 16),
    "Tarde Pico (17h–19h)": (17, 19),
    "Noite (20h–22h)": (20, 22),
    "Noite Tardia (23h)": (23, 23),
}

def get_faixa(hora: int) -> str:
    for nome, (inicio, fim) in FAIXAS.items():
        if inicio <= hora <= fim:
            return nome
    return "Outra"

df["faixa_horaria"] = df["hora"].apply(get_faixa)

# Calculo dos intevalos entre as partidas

def calcular_intervalos(grupo: pd.DataFrame) -> pd.Series:

    minutos = grupo["minuto_dia"].sort_values().values
    if len(minutos) < 2:
        return pd.Series(dtype=float)
    intervalos = np.diff(minutos)
    # Remove dados inconsistentes
    intervalos = intervalos[intervalos > 0]
    return pd.Series(intervalos)




def calcular_irn(intervalos: np.ndarray) -> float:

    if len(intervalos) < 2:
        return np.nan
    media = np.mean(intervalos)
    if media == 0:
        return np.nan
    cv = np.std(intervalos, ddof=1) / media
    return round(1 / (1 + cv), 4)


chave_grupo = ["servico", "sentido", "data_dt", "faixa_horaria"]
registros = []

for (servico, sentido, data, faixa), grp in df.groupby(chave_grupo):
    intervalos = calcular_intervalos(grp)
    if len(intervalos) < 2:
        continue
    registros.append({
        "servico": servico,
        "sentido": sentido,
        "data": data.date(),
        "faixa_horaria": faixa,
        "n_partidas": len(grp),
        "n_intervalos": len(intervalos),
        "intervalo_medio_min": round(np.mean(intervalos), 2),
        "intervalo_dp_min": round(np.std(intervalos, ddof=1), 2),
        "intervalo_min_min": round(np.min(intervalos), 2),
        "intervalo_max_min": round(np.max(intervalos), 2),
        "IRN_dia": calcular_irn(intervalos.values),
    })

df_irn_diario = pd.DataFrame(registros)




df_resumo = (
    df_irn_diario
    .groupby(["servico", "sentido", "faixa_horaria"])
    .agg(
        dias_operados=("data", "nunique"),
        n_partidas_total=("n_partidas", "sum"),
        intervalo_medio_min=("intervalo_medio_min", "mean"),
        intervalo_dp_medio=("intervalo_dp_min", "mean"),
        IRN=("IRN_dia", "mean"),
    )
    .round(4)
    .reset_index()
)


df_score_geral = (
    df_resumo
    .groupby(["servico", "sentido"])
    .apply(
        lambda x: pd.Series({
            "faixas_com_dados": len(x),
            "IRN_geral": round(
                np.average(x["IRN"], weights=x["dias_operados"]), 4
            ),
            "intervalo_medio_global": round(x["intervalo_medio_min"].mean(), 2),
            "dias_operados_total": x["dias_operados"].sum(),
        })
    )
    .reset_index()
    .sort_values("IRN_geral", ascending=False)
)

# Classificação qualitativa

def classificar_irn(irn: float) -> str:
    if pd.isna(irn):
        return "Sem dados"
    if irn >= 0.80:
        return "Excelente"
    if irn >= 0.65:
        return "Bom"
    if irn >= 0.50:
        return "Regular"
    if irn >= 0.35:
        return "Ruim"
    return "Crítico"

df_score_geral["classificacao"] = df_score_geral["IRN_geral"].apply(classificar_irn)
df_resumo["classificacao"] = df_resumo["IRN"].apply(classificar_irn)




print("RANKING GERAL DAS LINHAS POR IRN (Top 20 melhores)")
print(df_score_geral.head(20).to_string(index=False))
print("\n")
print("RANKING GERAL DAS LINHAS POR IRN (10 piores)")
print(df_score_geral.tail(10).to_string(index=False))
print("\n")
print("DISTRIBUIÇÃO DAS CLASSIFICAÇÕES (todas as linhas × sentidos)")
print("\n")
dist = df_score_geral["classificacao"].value_counts()
total = len(df_score_geral)


print("IRN MÉDIO POR FAIXA HORÁRIA (consolidado todas as linhas)")
faixa_media = (
    df_resumo
    .groupby("faixa_horaria")["IRN"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "IRN_medio", "std": "IRN_dp", "count": "n_linhas"})
    .sort_values("IRN_medio", ascending=False)
    .round(4)
)
print(faixa_media.to_string())
print("\n")



OUTPUT_RESUMO  = "regularidade_por_faixa.csv"
OUTPUT_SCORE   = "score_geral_linhas.csv"
OUTPUT_DIARIO  = "irn_diario_detalhado.csv"

df_resumo.to_csv(OUTPUT_RESUMO, index=False, encoding="utf-8-sig")
df_score_geral.to_csv(OUTPUT_SCORE, index=False, encoding="utf-8-sig")
df_irn_diario.to_csv(OUTPUT_DIARIO, index=False, encoding="utf-8-sig")



RANKING GERAL DAS LINHAS POR IRN (Top 20 melhores)
servico sentido  faixas_com_dados  IRN_geral  intervalo_medio_global  dias_operados_total classificacao
LECD105     Ida               1.0     1.0000                   16.00                  1.0     Excelente
    753     Ida               1.0     0.9892                   64.50                  1.0     Excelente
    771     Ida               1.0     0.9881                   58.50                  1.0     Excelente
    711     Ida               1.0     0.9766                   29.50                  1.0     Excelente
  SP918   Volta               1.0     0.9682                   43.00                  1.0     Excelente
    951     Ida               1.0     0.9484                   26.00                  1.0     Excelente
    393   Volta               1.0     0.9293                   46.50                  1.0     Excelente
 SPA613     Ida               1.0     0.9211                   33.00                  1.0     Excelente
    621   Vol

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

FILE_PATH       = "/content/viagem_informada.csv"
LIMIAR_BUMP_MIN = 3
PICO_MANHA      = (6,  9)
PICO_TARDE      = (17, 19)

def is_pico(hora: int) -> bool:
    return (PICO_MANHA[0] <= hora <= PICO_MANHA[1] or
            PICO_TARDE[0] <= hora <= PICO_TARDE[1])

FAIXAS = {
    "Madrugada (00h–05h)":        (0,  5),
    "Manhã Pico (06h–09h)":       (6,  9),
    "Manhã Fora-Pico (10h–11h)":  (10, 11),
    "Almoço (12h–13h)":           (12, 13),
    "Tarde Fora-Pico (14h–16h)":  (14, 16),
    "Tarde Pico (17h–19h)":       (17, 19),
    "Noite (20h–22h)":            (20, 22),
    "Noite Tardia (23h)":         (23, 23),
}

def get_faixa(hora: int) -> str:
    for nome, (ini, fim) in FAIXAS.items():
        if ini <= hora <= fim:
            return nome
    return "Outra"


df = pd.read_csv(FILE_PATH)
df["partida_dt"] = pd.to_datetime(df["partida"], format="%H:%M", errors="coerce")
df["data_dt"]    = pd.to_datetime(df["data"],    format="%d/%m/%Y", errors="coerce")
df = df.dropna(subset=["partida_dt", "data_dt"])

df["hora"]       = df["partida_dt"].dt.hour
df["minuto_dia"] = df["hora"] * 60 + df["partida_dt"].dt.minute

print(f"\n{len(df):,} registros | {df['servico'].nunique()} linhas | "
      f"{df['data_dt'].min().date()} → {df['data_dt'].max().date()}\n")


def detectar_grupos(minutos: np.ndarray, veiculos: list, limiar: int) -> list:
    grupos = []
    i = 0
    while i < len(minutos):
        grupo_min  = [minutos[i]]
        grupo_veic = [veiculos[i]]

        while i + 1 < len(minutos):
            intervalo = int(minutos[i + 1] - minutos[i])
            if intervalo < 0:
                break
            if intervalo <= limiar:
                i += 1
                grupo_min.append(minutos[i])
                grupo_veic.append(veiculos[i])
            else:
                break

        if len(grupo_min) >= 2:
            grupos.append({
                "partidas":    grupo_min,
                "veiculos":    grupo_veic,
                "tamanho":     len(grupo_min),
                "duracao_min": int(grupo_min[-1] - grupo_min[0]),
                "inicio_min":  grupo_min[0],
                "fim_min":     grupo_min[-1],
            })
        i += 1

    return grupos


grupos_total = []

for (servico, sentido, data), grp in df.groupby(["servico", "sentido", "data_dt"]):
    grp_ord  = grp.sort_values("minuto_dia").reset_index(drop=True)
    minutos  = grp_ord["minuto_dia"].values.astype(int)
    veiculos = (grp_ord["id_veiculo"].tolist()
                if "id_veiculo" in grp_ord else ["?"] * len(grp_ord))

    for g in detectar_grupos(minutos, veiculos, LIMIAR_BUMP_MIN):
        hora_inicio = g["inicio_min"] // 60
        pico        = is_pico(hora_inicio)

        horarios = " → ".join(
            f"{m // 60:02d}:{m % 60:02d}" for m in g["partidas"]
        )

        grupos_total.append({
            "servico":      servico,
            "sentido":      sentido,
            "data":         data.date(),
            "tamanho":      g["tamanho"],
            "duracao_min":  g["duracao_min"],
            "horarios":     horarios,
            "veiculos":     " → ".join(str(v) for v in g["veiculos"]),
            "hora_inicio":  hora_inicio,
            "faixa":        get_faixa(hora_inicio),
            "em_pico":      pico,
            "veredicto":    "JUSTIFICÁVEL" if pico else "NÃO JUSTIFICÁVEL",
        })

df_grupos = pd.DataFrame(grupos_total)


total        = len(df_grupos)
justif       = (df_grupos["veredicto"] == "JUSTIFICÁVEL").sum()
nao_justif   = (df_grupos["veredicto"] == "NÃO JUSTIFICÁVEL").sum()
pares        = (df_grupos["tamanho"] == 2).sum()
trios_mais   = (df_grupos["tamanho"] >= 3).sum()
maior_grupo  = df_grupos["tamanho"].max()

print(f"{total:,} grupos de bumping detectados")
print(f"Justificáveis (pico)    : {justif:>5,}")
print(f"Não justificáveis       : {nao_justif:>5,}")
print(f"Pares (2 ônibus)        : {pares:>5,}")
print(f"Trios ou mais (≥3)      : {trios_mais:>5,}")
print(f"Maior grupo detectado   : {maior_grupo:>5} ônibus")


df_grupos["contribuicao"] = df_grupos.apply(
    lambda r: r["tamanho"] ** 2 * (1 if r["em_pico"] else 3), axis=1
)

viagens_por_linha = (
    df.groupby(["servico", "sentido"])
    .size()
    .reset_index(name="total_viagens")
)

agg = (
    df_grupos
    .groupby(["servico", "sentido", "veredicto"])
    .agg(
        n_grupos=("tamanho", "count"),
        onibus_afetados=("tamanho", "sum"),
        maior_grupo=("tamanho", "max"),
        score_parcial=("contribuicao", "sum"),
    )
    .reset_index()
)

agg_pivot = agg.pivot_table(
    index=["servico", "sentido"],
    columns="veredicto",
    values=["n_grupos", "onibus_afetados", "score_parcial"],
    fill_value=0,
).reset_index()

agg_pivot.columns = [
    "_".join(filter(None, col)).strip() if col[1] else col[0]
    for col in agg_pivot.columns
]

for prefixo in ["n_grupos", "onibus_afetados", "score_parcial"]:
    for suf in ["JUSTIFICÁVEL", "NÃO JUSTIFICÁVEL"]:
        col = f"{prefixo}_{suf}"
        if col not in agg_pivot.columns:
            agg_pivot[col] = 0

maior_por_linha = (
    df_grupos.groupby(["servico", "sentido"])["tamanho"].max()
    .reset_index().rename(columns={"tamanho": "maior_grupo"})
)

df_rank = (
    agg_pivot
    .merge(maior_por_linha, on=["servico", "sentido"], how="left")
    .merge(viagens_por_linha, on=["servico", "sentido"], how="left")
)

df_rank["total_grupos"]      = (df_rank["n_grupos_JUSTIFICÁVEL"] +
                                 df_rank["n_grupos_NÃO JUSTIFICÁVEL"])
df_rank["total_onibus"]      = (df_rank["onibus_afetados_JUSTIFICÁVEL"] +
                                 df_rank["onibus_afetados_NÃO JUSTIFICÁVEL"])
df_rank["score_severidade"]  = (df_rank["score_parcial_JUSTIFICÁVEL"] +
                                 df_rank["score_parcial_NÃO JUSTIFICÁVEL"])
df_rank["taxa_bump_pct"]     = (df_rank["total_onibus"] /
                                 df_rank["total_viagens"] * 100).round(2)


df_rank = df_rank.sort_values("score_severidade", ascending=False).reset_index(drop=True)

def classificar(score):
    if score == 0:    return "Sem bumping"
    if score <= 6:    return "Leve"
    if score <= 20:   return "Moderado"
    if score <= 60:   return "Grave"
    return "Crítico"

df_rank["classificacao"] = df_rank["score_severidade"].apply(classificar)

# Print sem limite de linhas
cols = [
    "servico", "sentido",
    "n_grupos_JUSTIFICÁVEL", "n_grupos_NÃO JUSTIFICÁVEL",
    "total_grupos", "maior_grupo",
    "total_onibus", "taxa_bump_pct",
    "score_severidade", "classificacao",
]
print("TODAS AS LINHAS — RANKING DE SEVERIDADE")
print("score = somatório (tamanho_grupo^2 × peso)  |  peso: fora_pico=3, pico=1")
pd.set_option("display.max_rows", None)
print(df_rank[cols].to_string(index=False))
pd.reset_option("display.max_rows")

print("\n")
print("GRUPOS DE ≥ 3 ÔNIBUS (os mais graves)")
trios = (
    df_grupos[df_grupos["tamanho"] >= 3]
    .sort_values(["tamanho", "duracao_min"], ascending=[False, False])
    [["servico", "sentido", "data", "tamanho", "duracao_min",
      "horarios", "faixa", "veredicto"]]
)
print(trios.to_string(index=False))

print("\n")
print("BUMPING POR FAIXA HORÁRIA")

faixa_agg = (
    df_grupos
    .groupby(["faixa", "veredicto"])
    .agg(n_grupos=("tamanho", "count"), onibus=("tamanho", "sum"))
    .reset_index()
    .pivot_table(index="faixa", columns="veredicto",
                 values=["n_grupos", "onibus"], fill_value=0)
)
faixa_agg.columns = ["_".join(c) for c in faixa_agg.columns]
for col in ["n_grupos_JUSTIFICÁVEL", "n_grupos_NÃO JUSTIFICÁVEL",
            "onibus_JUSTIFICÁVEL",   "onibus_NÃO JUSTIFICÁVEL"]:
    if col not in faixa_agg.columns:

            faixa_agg[col] = 0

faixa_agg["total_grupos"] = (faixa_agg["n_grupos_JUSTIFICÁVEL"] +

                              faixa_agg["n_grupos_NÃO JUSTIFICÁVEL"])
print(faixa_agg.sort_values("total_grupos", ascending=False).to_string())

print("\n")

df_grupos.drop(columns=["contribuicao"]).to_csv(
    "bus_bumping_grupos.csv", index=False, encoding="utf-8-sig")

df_rank.to_csv(
    "bus_bumping_ranking_linhas.csv", index=False, encoding="utf-8-sig")




256,725 registros | 549 linhas | 2024-04-11 → 2026-04-16

365 grupos de bumping detectados
Justificáveis (pico)    :   155
Não justificáveis       :   210
Pares (2 ônibus)        :   362
Trios ou mais (≥3)      :     3
Maior grupo detectado   :     3 ônibus
TODAS AS LINHAS — RANKING DE SEVERIDADE
score = somatório (tamanho_grupo^2 × peso)  |  peso: fora_pico=3, pico=1
servico sentido  n_grupos_JUSTIFICÁVEL  n_grupos_NÃO JUSTIFICÁVEL  total_grupos  maior_grupo  total_onibus  taxa_bump_pct  score_severidade classificacao
    864   Volta                    5.0                       22.0          27.0            2          54.0           3.64             284.0       Crítico
    838   Volta                    7.0                        9.0          16.0            2          32.0           2.36             136.0       Crítico
    864     Ida                    5.0                        9.0          14.0            3          29.0           2.12             133.0       Crítico
    371     

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap
import re

df_reg_faixa = pd.read_csv('regularidade_por_faixa.csv')
df_bumping = pd.read_csv('bus_bumping_grupos.csv')

def extrair_hora_inicial(faixa_str):

    match = re.search(r'\((\d{2})h', str(faixa_str))
    return int(match.group(1)) if match else 99

def gerar_dashboard(linha):
    linha = str(linha).upper()

    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f'Análise Operacional (Ordenada) - Linha {linha}', fontsize=18, weight='bold')


    df_linha_reg = df_reg_faixa[df_reg_faixa['servico'].astype(str).str.upper() == linha]

    if not df_linha_reg.empty:
        pivot_reg = df_linha_reg.pivot_table(index='sentido', columns='faixa_horaria', values='IRN', aggfunc='mean')

        colunas_ordenadas = sorted(pivot_reg.columns, key=extrair_hora_inicial)
        pivot_reg = pivot_reg[colunas_ordenadas]

        sns.heatmap(pivot_reg, annot=True, cmap='RdYlGn', vmin=0, vmax=1, fmt=".3f",
                    linewidths=.5, ax=axes[0], cbar_kws={'label': 'Índice de Regularidade (IRN)'})
        axes[0].set_title('Mapa de Calor de Regularidade (IRN) por Faixa Horária', fontsize=14)
        axes[0].set_xlabel('Faixa Horária', fontsize=12)
        axes[0].set_ylabel('Sentido', fontsize=12)

        labels = [textwrap.fill(item.get_text(), 15) for item in axes[0].get_xticklabels()]
        axes[0].set_xticklabels(labels, rotation=45)
        axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0)
    else:
        axes[0].set_title('Sem dados de regularidade por faixa para esta linha.')


    bumping_linha = df_bumping[df_bumping['servico'].astype(str).str.upper() == linha]
    score_linha = bumping_linha.groupby('faixa').size().reset_index(name='score_linha')

    bumping_todas = df_bumping.groupby(['servico', 'faixa']).size().reset_index(name='eventos')
    media_geral = bumping_todas.groupby('faixa')['eventos'].mean().reset_index(name='media_geral')

    df_plot_bumping = pd.merge(media_geral, score_linha, on='faixa', how='left').fillna({'score_linha': 0})

    if not df_plot_bumping.empty:
        df_plot_bumping['hora_ordem'] = df_plot_bumping['faixa'].apply(extrair_hora_inicial)
        df_plot_bumping = df_plot_bumping.sort_values('hora_ordem')

        df_melt = df_plot_bumping.melt(id_vars=['faixa', 'hora_ordem'], value_vars=['score_linha', 'media_geral'],
                                       var_name='Tipo', value_name='Score (Qtd. Eventos)')

        df_melt['Tipo'] = df_melt['Tipo'].map({
            'score_linha': f'Linha {linha}',
            'media_geral': 'Média Geral (Todas as Linhas)'
        })

        sns.barplot(data=df_melt, x='faixa', y='Score (Qtd. Eventos)', hue='Tipo',
                    ax=axes[1], palette=['#1f77b4', '#ff7f0e'])

        axes[1].set_title('Comparativo de Score de Bumping por Faixa Horária', fontsize=14)
        axes[1].set_xlabel('Faixa Horária', fontsize=12)
        axes[1].set_ylabel('Score / Eventos', fontsize=12)

        labels2 = [textwrap.fill(item.get_text(), 12) for item in axes[1].get_xticklabels()]
        axes[1].set_xticklabels(labels2, rotation=0)
    else:
        axes[1].set_title('Sem dados de bumping comparativos.')

    plt.tight_layout()
    nome_arquivo = f'dashboard_{linha}.png'
    plt.savefig(nome_arquivo, dpi=300)
    plt.close()
    print(f"\nDashboard gerado com sucesso e salvo como '{nome_arquivo}'.")

linha = input("insira uma linha de ônibus: ")
gerar_dashboard(linha)
# O sistema retorna as visualizações de acordo com a disponibilidade de dados na Base de dados original.
# Para linhas com uma farta disponibilidade de dados o sistema funciona muito bem
#linha de exemplo : "864"


KeyboardInterrupt: Interrupted by user